# neuralCAD-Edit

neuralCAD-Edit is stored as a mongita database containing metadata and filepaths, which point to objects (images, videos, .step etc.) in the file tree. 

This notebook contains some ways to visualise the data. We recommend always accessing the data through the database, and the access patterns used here should give you and idea about how to do this.

### Imports and config

Make sure the paths in the config are correct

In [ ]:
import os
import os.path as osp
import sys
module_path = os.path.abspath(os.path.join('../..'))
sys.path.append(module_path)
from src.utils.db import DatabaseManager
from src.utils.process_config import load_config
from IPython.display import display, Video
from PIL import Image as PILImage, ImageDraw, ImageFont
import shutil

config_path = os.path.join(module_path, 'src', 'config', 'edit_192_external.json')

class Args:
    def __init__(self, config):
        self.config = config
args = Args(config=config_path)
config = load_config(args.config)
dbm = DatabaseManager(config)
print("Database manager initialized from config:", config_path)
dbm.print_db_schema_counts()



### Function defs

Just run. You can collapse to get it out of the way.

In [ ]:
def display_images_side_by_side(image_paths, new_height=512, labels=None, n_horizontal=4):
    """
    Display a list of local image paths side by side with optional labels above each image.
    If more images are provided than the n_horizontal limit, they will wrap to the next line.

    Args:
        image_paths (list): List of file paths to images
        new_height (int): Height to resize images to
        labels (list, optional): List of text labels to display above each image
        n_horizontal (int): Maximum number of images per row
    """
    
    # Load and resize images
    images = []
    for img_path in image_paths:
        if os.path.exists(img_path):
            try:
                img = PILImage.open(img_path)
                # Calculate new width to maintain aspect ratio
                width_ratio = new_height / img.height
                new_width = int(img.width * width_ratio)
                img = img.resize((new_width, new_height))
                images.append(img)
            except Exception as e:
                print(f"Error loading image {img_path}: {e}")
    
    if not images:
        print("No valid images found to display")
        return
    
    # Calculate grid dimensions
    n_rows = (len(images) + n_horizontal - 1) // n_horizontal  # Ceiling division
    
    # Group images into rows
    image_rows = []
    label_rows = []
    
    for row_idx in range(n_rows):
        start_idx = row_idx * n_horizontal
        end_idx = min(start_idx + n_horizontal, len(images))
        
        row_images = images[start_idx:end_idx]
        row_labels = labels[start_idx:end_idx] if labels else None
        
        image_rows.append(row_images)
        label_rows.append(row_labels)
    
    # Calculate dimensions for the combined image
    label_height = 30 if labels else 0
    row_height = new_height + label_height
    
    # Find the maximum width needed (widest row)
    max_row_width = 0
    for row_images in image_rows:
        row_width = sum(img.width for img in row_images)
        max_row_width = max(max_row_width, row_width)
    
    total_height = row_height * n_rows
    
    # Create the combined image
    combined_image = PILImage.new('RGB', (max_row_width, total_height), color=(255, 255, 255))
    draw = ImageDraw.Draw(combined_image)
    
    try:
        font = ImageFont.truetype("Arial", 14)
    except IOError:
        font = ImageFont.load_default()
    
    # Process each row
    for row_idx, (row_images, row_labels) in enumerate(zip(image_rows, label_rows)):
        current_y = row_idx * row_height
        current_x = 0
        
        for i, img in enumerate(row_images):
            # Paste image
            combined_image.paste(img, (current_x, current_y + label_height))
            
            # Add label if provided
            if row_labels and i < len(row_labels):
                text = str(row_labels[i])
                text_width = draw.textlength(text, font=font)
                text_x = current_x + (img.width - text_width) // 2
                text_y = current_y + 5
                draw.text((text_x, text_y), text, fill=(0, 0, 0), font=font)
            
            current_x += img.width
    
    # Display the combined image
    display(combined_image)

def display_video_and_transcript(video_path, transcript):
    """
    Display a video and its transcript.
    
    Args:
        video_path (str): Path to the video file
        transcript (str): Transcript text to display below the video
    """
    if os.path.exists(video_path):
        print(f"Displaying video: {video_path}")
        display(Video(video_path, embed=True))
        print("Transcript:")
        for k, v in transcript.items():
            print(f"{k}: {v}")
    else:
        print(f"Video file {video_path} does not exist.")
    
def display_request_info(dbm, request_id):
    request = dbm.requests.find_one({"_id": request_id})
    if request_id is None:
        print(f"No request found with ID: {request_id}")
        return
    
    print("Request details:")
    print(f"Request ID: {request_id}")
    print(f"Keys: {list(request.keys())}")

    print()
    print("Brep:")
    breps = dbm.breps.find_one({"_id": request["brep_start"]})
    for key, value in breps.items():
        if value:
            if len(str(value)) > 500:
                value = '...'
            print(f"{key}: {value}")

    print()
    print("User details:")
    user = dbm.users.find_one({"_id": request["user"]})
    for key, value in user.items():
        if value:
            print(f"{key}: {value}")

    # show the video with key 30_720_audio
    print()
    video_key = "30_720_audio"

    if video_key in request:
        video_path = os.path.join(dbm.root_dir, request[video_key])
        transcription = request.get("transcript", {})
        display_video_and_transcript(video_path, transcription)

def display_edit_info(dbm, edit_id, n_to_display=4):
    edit = dbm.edits.find_one({"_id": edit_id})
    if edit_id is None:
        print(f"No edit found with ID: {edit_id}")
        return
    print("Edit details:")
    print(f"Edit ID: {edit_id}")
    print(f"Keys: {list(edit.keys())}")

    print()
    print("Events")
    print(f"Number of events: {len(edit['events'])}")
    # pick n_to_display uniformly from the events
    if len(edit['events']) > n_to_display:
        step = len(edit['events']) // n_to_display
        events_to_display = edit['events'][::step][:n_to_display]
    else:
        events_to_display = edit['events']
    for event in events_to_display:
        print(event)

    print()
    print("brep end:")
    breps = dbm.breps.find_one({"_id": edit["brep_end"]})
    for key, value in breps.items():
        if value:
            if len(str(value)) > 500:
                value = '...'
            print(f"{key}: {value}")

    # show the toprightiso image of the brep end
    brep_images = dbm.get_brep_images(edit["brep_end"], views=["toprightiso"], format=["jpg", "png"])
    if brep_images:
        brep_image_path = os.path.join(dbm.root_dir, brep_images[0])
        display_images_side_by_side([brep_image_path])

    print()
    print("Frames:")
    frames_dir = os.path.join(dbm.root_dir, edit["frames_dir"])
    if os.path.exists(frames_dir):
        print(f"Frames directory found at: {frames_dir}")
        frames = os.listdir(frames_dir)
        frames.sort()
        display_images_side_by_side([os.path.join(frames_dir, f) for f in frames])
    else:
        print(f"Frames directory not found at: {frames_dir}")

    print()
    print("Cots:")
    cot_entries = list(dbm.cots.find({"edit": edit_id}))
    for cot in cot_entries:
        user = dbm.users.find_one({"_id": cot["user"]})
        user_name = user["_id"] if user else "Unknown"
        print(f"CoT by {user_name}:\n {cot['cot']}")

def display_request_and_edits(dbm, request_id, output_dir=None, views=["toprightiso", "sketch", "render"], n_horizontal=4, height=384, models=[]):
    request = dbm.requests.find_one({"_id": request_id})
    edits = dbm.edits.find({"request": request_id})

    request_id = request["_id"]
    print(f"Request ID: {request_id}")
    user_2_img = {}
    for edit in edits:
        brep = dbm.breps.find_one({"_id": edit["brep_end"]})
        brep_id = brep["_id"]
        brep_img = dbm.get_brep_images(brep_id, views=views)
        if brep_img:
            brep_img = brep_img[0]
        else:
            continue
        brep_img = os.path.join(dbm.root_dir, brep_img)

        user = dbm.users.find_one({"_id": edit["user"]})

        if edit["user"] == request["user"]:
            display_user = "Human GT"
            # user_2_img["Human GT"] = brep_img
        elif user["is_human"]:
            display_user = "Human Other"
            # user_2_img["Human Other"] = brep_img
        else:
            display_user = user["_id"]
            # user_2_img[user["_id"]] = brep_img
        
        if len(models) > 0 and display_user not in models:
            continue


        user_2_img[display_user] = brep_img
        print(f"Edit ID: {edit['_id']}, User: {user['_id']}, BRep Image: {brep_img}")

    # display the request video
    video_key = "30_720_audio"
    if video_key in request:
        video_path = os.path.join(dbm.root_dir, request[video_key])
        transcription = request.get("transcript", {})
        display_video_and_transcript(video_path, transcription)

    # display the request brep images
    brep_start = dbm.breps.find_one({"_id": request["brep_start"]})
    if brep_start:
        request_images = dbm.get_brep_images(brep_start["_id"], views=views, format=["jpg", "png"])
        
        request_images = [os.path.join(dbm.root_dir, img) for img in request_images]
        display_images_side_by_side(
            request_images,
            labels=[""] * len(request_images),
            new_height=height
        )
        if output_dir:
            for idx, img_path in enumerate(request_images):
                basename = os.path.basename(img_path)
                output_folder = os.path.join(output_dir, request_id)
                os.makedirs(output_folder, exist_ok=True)
                output_path = os.path.join(output_folder, basename)
                print(img_path, "->", output_path)
                shutil.copy(img_path, output_path)

    # sort the user_2_img dictionary by user name
    user_2_img = dict(sorted(user_2_img.items(), key=lambda item: item[0]))


    # if a list of models is provided, filter the user_2_img to only include those models (if they exist)
    if models:
        user_2_img = {user: img for user, img in user_2_img.items() if user in models}



    labels, images = [], []
    for user, img in user_2_img.items():
        labels.append(user)
        images.append(img)
    display_images_side_by_side(images, labels=labels, n_horizontal=n_horizontal, new_height=height)
    if output_dir:
        for img_path, user in zip(images, labels):
            basename = os.path.basename(img_path)
            output_folder = os.path.join(output_dir, request_id, user)
            os.makedirs(output_folder, exist_ok=True)
            output_path = os.path.join(output_folder, basename)
            print(img_path, "->", output_path)
            shutil.copy(img_path, output_path)


### Display all information about a single request
Here, we find a request from the database, which is from an medium difficuly draw-static request on a parametric assembly task.


In [ ]:
request_ids = dbm.requests.find({"modality": "draw-static", "difficulty": "medium", "assembly": True, "parametric": True})
request_ids = [request["_id"] for request in request_ids]

display_request_info(dbm, request_ids[0] if request_ids else None)

### Display all information about a single edit

Here, we take a request, and find an edit which corresponds to that request.

In [ ]:
request_ids = dbm.requests.find({"modality": "draw-static", "difficulty": "medium", "assembly": True, "parametric": True})
request_ids = [request["_id"] for request in request_ids]

edit_ids = dbm.edits.find({"request": {"$in": request_ids}})
edit_ids = [edit["_id"] for edit in edit_ids]

display_edit_info(dbm, edit_ids[0])

### Display a request and the output of different users/AIs

Four requests are shown - these are the ones in the example video. If an AI did not produce a valid CAD model, then nothing is shown.

In [ ]:
# request ids from example video
request_ids = [
    # draw-static
    "F332D3FXML85WLR2_1770192609.750378",
    # draw-temp
    "B7A2N74ZJBF9MZHU_1770251861.898911",
    # interactive
    "3YH2WFSRM22W7DKT_1769160931.493333",
    # text
    "SUJ2G2UMJQR7PMBX_1759209987.785593",
]

models = [
    "Human GT",
    "Human Other",
    'gemini-3-pro_cadquery-script',
    "claude-sonnet-4.5_cadquery-script",
    "gpt-5.2_cadquery-script",
]

for i in range(len(request_ids)):
    display_request_and_edits(dbm, request_ids[i] if request_ids else None, output_dir=None, views=["dslrender","sketch", "render", "toprightiso"], n_horizontal=5, height=512, models=models)

### Create video of human edits

Creates video of the ground truth (where the edit user matches the request user) and human baseline (where the edit user is human, but does not match the request user). This is done for the four requests in the example video.

There are lots of functions at the top of the cell to keep it contained. Just go down to the bottom to do your request selection.

In [ ]:
"""
Create a video from images keyed by absolute timeline seconds, with optional burned-in subtitles.

Inputs:
  - images_ts: {timestamp_seconds: image_path}
  - subs_ts (optional): {timestamp_seconds: subtitle_text}

Behavior:
  - Images are shown from their timestamp until the next image timestamp.
  - If the first image timestamp is > 0, the first image is held from 0 to that timestamp
    (so the video always starts at t=0 with content).
  - Last image holds for tail_duration seconds.
  - speedup > 1.0 makes the video play faster by dividing all times/durations by speedup.
  - Subtitles last until the next subtitle, or subtitle_tail_duration for the last.
  - Subtitles are burned-in via ffmpeg subtitles filter (SRT).
"""

from __future__ import annotations

import os
import shlex
import subprocess
import tempfile
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any


@dataclass(frozen=True)
class FrameSpec:
    path: str
    start: float
    duration: float


def _run(cmd: List[str]) -> None:
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if p.returncode != 0:
        raise RuntimeError(
            "FFmpeg failed.\n"
            f"Command: {' '.join(shlex.quote(c) for c in cmd)}\n\n"
            f"STDOUT:\n{p.stdout}\n\nSTDERR:\n{p.stderr}\n"
        )


def _probe_image_size(path: str) -> Tuple[int, int]:
    cmd = [
        "ffprobe", "-v", "error",
        "-select_streams", "v:0",
        "-show_entries", "stream=width,height",
        "-of", "csv=s=x:p=0",
        path,
    ]
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if p.returncode != 0 or not p.stdout.strip():
        raise RuntimeError(f"ffprobe failed for {path}:\n{p.stderr}")
    w, h = p.stdout.strip().split("x")
    return int(w), int(h)


def _sorted_time_to_path(images_ts: Dict[float, str]) -> List[Tuple[float, str]]:
    if not images_ts:
        raise ValueError("images_ts is empty.")

    items: List[Tuple[float, str]] = []
    for t, path in images_ts.items():
        try:
            tf = float(t)
        except Exception as e:
            raise ValueError(f"Invalid timestamp key: {t!r}") from e
        if tf < 0:
            raise ValueError(f"Negative timestamp not allowed: {tf}")

        path_s = str(path)
        if not Path(path_s).exists():
            raise FileNotFoundError(path_s)

        items.append((tf, path_s))

    items.sort(key=lambda kv: kv[0])

    # Ensure strictly increasing timestamps (dict keys already unique, but still validate numeric ordering)
    for (t1, _), (t2, _) in zip(items, items[1:]):
        if t2 <= t1:
            raise ValueError("Image timestamps must be strictly increasing (no duplicates).")

    return items


def _build_frames(
    images_ts: Dict[float, str],
    *,
    tail_duration: float,
    speedup: float,
    hold_first_from_zero: bool = True,
) -> List[FrameSpec]:
    if speedup <= 0:
        raise ValueError("speedup must be > 0.")
    if tail_duration <= 0:
        raise ValueError("tail_duration must be > 0.")

    items = _sorted_time_to_path(images_ts)  # [(t, path)]
    frames: List[FrameSpec] = []

    first_t, first_path = items[0]
    if hold_first_from_zero and first_t > 0:
        # show first image from t=0 up to its first timestamp
        frames.append(FrameSpec(path=first_path, start=0.0, duration=(first_t / speedup)))

    # show each image until next image
    for (t1, p1), (t2, _) in zip(items, items[1:]):
        dur = (t2 - t1) / speedup
        if dur <= 0:
            raise ValueError(f"Non-positive duration between {t1} and {t2}.")
        frames.append(FrameSpec(path=p1, start=t1 / speedup, duration=dur))

    # last image tail
    last_t, last_path = items[-1]
    frames.append(FrameSpec(path=last_path, start=last_t / speedup, duration=tail_duration / speedup))
    return frames


def _write_concat_list(frames: List[FrameSpec], concat_path: str) -> None:
    lines: List[str] = []
    last_abs: Optional[str] = None

    for f in frames:
        abs_path = str(Path(f.path).resolve())
        last_abs = abs_path
        safe = abs_path.replace("'", r"'\''")
        lines.append(f"file '{safe}'")
        lines.append(f"duration {f.duration:.6f}")

    # Repeat last file to ensure the final frame appears reliably
    if last_abs is not None:
        safe_last = last_abs.replace("'", r"'\''")
        lines.append(f"file '{safe_last}'")

    Path(concat_path).write_text("\n".join(lines) + "\n", encoding="utf-8")


def _srt_timestamp(seconds: float) -> str:
    if seconds < 0:
        seconds = 0.0
    ms_total = int(round(seconds * 1000.0))
    hh = ms_total // 3_600_000
    ms_total -= hh * 3_600_000
    mm = ms_total // 60_000
    ms_total -= mm * 60_000
    ss = ms_total // 1000
    ms = ms_total - ss * 1000
    return f"{hh:02d}:{mm:02d}:{ss:02d},{ms:03d}"


def _write_srt(
    subs_ts: Dict[float, str],
    *,
    srt_path: str,
    video_duration: float,
    min_subtitle_duration: float = 2.0,
    max_subtitle_duration: float = 20.0,
    subtitle_duration: float | None = None,
) -> None:
    """
    Write subtitles to SRT.

    If subtitle_duration is provided:
        - Every subtitle gets exactly that duration
          (clamped to video_duration)
        - min/max_subtitle_duration are ignored

    Otherwise:
        - Subtitle lasts until next subtitle
        - Clamped between min_subtitle_duration and max_subtitle_duration
    """

    if not subs_ts:
        Path(srt_path).write_text("", encoding="utf-8")
        return

    items = sorted((float(t), str(text)) for t, text in subs_ts.items())

    blocks = []
    idx = 1

    for i, (start, text) in enumerate(items):
        if not text.strip():
            continue

        if subtitle_duration is not None:
            # --- Fixed duration override ---
            end = start + subtitle_duration

        else:
            # --- Original dynamic behaviour ---
            if i < len(items) - 1:
                end = items[i + 1][0]
            else:
                end = start + max_subtitle_duration

            raw_duration = end - start
            duration = max(min_subtitle_duration, min(max_subtitle_duration, raw_duration))
            end = start + duration

        # Clamp to video end
        if end > video_duration:
            end = video_duration

        # Skip zero/negative duration
        if end <= start:
            continue

        blocks.append(
            f"{idx}\n"
            f"{_srt_timestamp(start)} --> {_srt_timestamp(end)}\n"
            f"{text.strip()}\n"
        )
        idx += 1

    Path(srt_path).write_text("\n".join(blocks).strip() + "\n", encoding="utf-8")

def _escape_drawtext_text(text: str) -> str:
    # Escape for ffmpeg drawtext text=...
    # - backslash first
    # - then single quote
    # - then colon (ffmpeg uses : as option separator)
    return (
        text.replace("\\", r"\\")
            .replace("'", r"\'")
            .replace(":", r"\:")
    )

def dict_seconds_to_video(
    images_ts: Dict[float, str],
    output_path: str,
    *,
    fps: int = 30,
    tail_duration: float = 2.0,
    speedup: float = 1.0,
    subs_ts: Optional[Dict[float, str]] = None,
    min_subtitle_duration: float = 2.0,
    max_subtitle_duration: float = 20.0,
    subtitle_duration: float | None = None,
    burn_in_text: str | None = None,
    burn_in_text_size: int = 24,
    width: int | None = None,
    height: int | None = None,
    crf: int = 18,
    preset: str = "medium",
    hold_first_from_zero: bool = True,
) -> None:
    frames = _build_frames(
        images_ts,
        tail_duration=tail_duration,
        speedup=speedup,
        hold_first_from_zero=hold_first_from_zero,
    )
    video_duration = sum(f.duration for f in frames)

    if width is None or height is None:
        w0, h0 = _probe_image_size(frames[0].path)
        width = width or w0
        height = height or h0

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    with tempfile.TemporaryDirectory() as td:
        concat_file = os.path.join(td, "concat.txt")
        _write_concat_list(frames, concat_file)

        srt_file = None
        if subs_ts:
            srt_file = os.path.join(td, "subs.srt")
            _write_srt(
                subs_ts,
                srt_path=srt_file,
                video_duration=video_duration,
                min_subtitle_duration=min_subtitle_duration,
                max_subtitle_duration=max_subtitle_duration,
                subtitle_duration=subtitle_duration,
            )

        vf_parts = [
            f"scale=w={width}:h={height}:force_original_aspect_ratio=decrease",
            f"pad={width}:{height}:(ow-iw)/2:(oh-ih)/2",
            f"fps={fps}",
            "format=yuv420p",
        ]

        if burn_in_text:
            safe_text = _escape_drawtext_text(burn_in_text)
            # Top-left: x=10, y=10. Box improves readability.
            vf_parts.append(
                "drawtext="
                f"text='{safe_text}':"
                "x=10:y=10:"
                f"fontsize={burn_in_text_size}:"
                "box=1:boxborderw=6"
            )

        if srt_file:
            vf_parts.append(f"subtitles={srt_file}")

        cmd = [
            "ffmpeg",
            "-y",
            "-hide_banner",
            "-loglevel", "error",
            "-f", "concat",
            "-safe", "0",
            "-i", concat_file,
            "-vf", ",".join(vf_parts),
            "-c:v", "libx264",
            "-preset", preset,
            "-crf", str(crf),
            "-movflags", "+faststart",
            output_path,
        ]

        try:
            _run(cmd)
            print(f"Video created at {output_path}")
        except Exception as e:
            raise RuntimeError(f"Failed to create video at {output_path}") from e





def normalize_epoch_tracks_with_image_durations(
    image_dict: Dict[float, Any],
    other_dicts: List[Dict[float, Any]],
    *,
    speedup: float = 1.0,
    min_image_duration: float = 0.1,
    max_image_duration: float = 10.0,
) -> Tuple[Dict[float, Any], List[Dict[float, Any]], float, float]:
    """
    Normalize multiple {epoch_timestamp: object} dicts into a shared timeline
    starting at t=0.0, then enforce min/max gaps *between images* and retime
    everything (images + subtitles) consistently.

    Inputs
    ------
    image_dict:
        {epoch: image_obj}  # frames
    other_dicts:
        [ {epoch: obj}, ... ]  # e.g. subtitles, other tracks
    speedup:
        >1.0 = play faster (shorter), <1.0 = slower. Linear scale on the
        original epochs before we enforce min/max image gaps.
    min_image_duration, max_image_duration:
        Clamp the time *between consecutive images* in the final timeline
        (in seconds). Only affects images directly, but subtitles are warped
        along with them to stay aligned.

    Behavior (high-level)
    ---------------------
    1. Find global minimum epoch across images + others = t0_epoch.
    2. Convert any epoch t to a base time:
           base(t) = (t - t0_epoch) / speedup
    3. Take sorted image base times, and build new image times by:
           gap_i = base_img[i+1] - base_img[i]
           gap_i' = clamp(gap_i, min_image_duration, max_image_duration)
       so new_img[0] == base_img[0], new_img[1] = new_img[0] + gap_0', ...
    4. Define a time-warp w(t) such that:
         - before the first image: times are unchanged
         - between each pair of images: everything is scaled linearly
           to fit the clamped gap
         - after the last image: everything is shifted by a constant
           (same offset as the last image)
       Then apply w() to both images and subtitles.
    5. This guarantees:
         - ordering is preserved
         - if a subtitle shared an epoch with an image, it still shares
           the same time after warping
         - subtitles remain “in the right place” relative to frames

    Returns
    -------
    norm_images: Dict[float, Any]
        {seconds_from_start: image_obj} after warping
    norm_others: List[Dict[float, Any]]
        Each dict is {seconds_from_start: obj} in the same warped timeline
    t0_epoch: float
        Epoch that became t = 0.0 before speedup
    timeline_end: float
        Time of the last image in the warped timeline
    """
    if speedup <= 0:
        raise ValueError("speedup must be > 0.")
    if min_image_duration <= 0 or max_image_duration <= 0:
        raise ValueError("min_image_duration and max_image_duration must be > 0.")
    if min_image_duration > max_image_duration:
        raise ValueError("min_image_duration cannot be greater than max_image_duration.")

    # --- 1. Find global minimum epoch ---
    all_ts: List[float] = []
    if image_dict:
        all_ts.extend(float(ts) for ts in image_dict.keys())
    for d in other_dicts:
        all_ts.extend(float(ts) for ts in d.keys())

    if not all_ts:
        # Nothing to normalize
        return {}, [dict() for _ in other_dicts], 0.0, 0.0

    t0_epoch = min(all_ts)

    def base_time(t_epoch: float) -> float:
        return (float(t_epoch) - t0_epoch) / speedup

    # --- 2. Image base times (seconds from start, with speedup) ---
    image_items = sorted(
        ((base_time(ts), img) for ts, img in image_dict.items()),
        key=lambda x: x[0],
    )
    img_base_times = [t for t, _ in image_items]
    img_objs = [img for _, img in image_items]

    if not img_base_times:
        raise ValueError("image_dict must not be empty; images define the timeline.")

    for t1, t2 in zip(img_base_times, img_base_times[1:]):
        if t2 <= t1:
            raise ValueError("Image base times must be strictly increasing.")

    # --- 3. Enforce min/max gaps between images (shift later ones only) ---
    warped_img_times: List[float] = [img_base_times[0]]  # keep first image as-is

    for i in range(1, len(img_base_times)):
        gap = img_base_times[i] - img_base_times[i - 1]
        if gap < min_image_duration:
            gap_clamped = min_image_duration
        elif gap > max_image_duration:
            gap_clamped = max_image_duration
        else:
            gap_clamped = gap

        warped_img_times.append(warped_img_times[i - 1] + gap_clamped)

    # Offsets for before-first and after-last
    delta_before = warped_img_times[0] - img_base_times[0]   # usually 0
    delta_after = warped_img_times[-1] - img_base_times[-1]

    # Precompute segment scales between images
    # Each segment: [base_start, base_end] -> [warp_start, warp_end]
    segments = []  # (base_start, base_end, warp_start, scale)
    for i in range(len(img_base_times) - 1):
        bs = img_base_times[i]
        be = img_base_times[i + 1]
        ws = warped_img_times[i]
        we = warped_img_times[i + 1]
        gap = be - bs
        scale = (we - ws) / gap if gap > 0 else 1.0
        segments.append((bs, be, ws, scale))

    def warp_time_from_base(t: float) -> float:
        """
        Map a base time (after t0 and speedup) into the warped timeline.
        - before first image: shift by delta_before
        - in between images: linear scale within the segment
        - after last image: shift by delta_after
        """
        if t <= img_base_times[0]:
            return t + delta_before
        if t >= img_base_times[-1]:
            return t + delta_after

        for (bs, be, ws, scale) in segments:
            if bs <= t <= be:
                return ws + (t - bs) * scale

        # Fallback (shouldn't happen)
        return t + delta_after

    # --- 4. Apply warp to images ---
    norm_images: Dict[float, Any] = {}
    for base_t, img in zip(img_base_times, img_objs):
        wt = warp_time_from_base(base_t)
        norm_images[wt] = img

    # --- 5. Apply warp to other tracks (e.g. subtitles) ---
    norm_others: List[Dict[float, Any]] = []
    for d in other_dicts:
        nd: Dict[float, Any] = {}
        for ts, obj in sorted(d.items(), key=lambda kv: kv[0]):
            bt = base_time(ts)
            wt = warp_time_from_base(bt)
            nd[wt] = obj
        norm_others.append(nd)

    timeline_end = warped_img_times[-1]
    return norm_images, norm_others, t0_epoch, timeline_end

def create_video_from_edit_id(dbm, edit_id):
    edit = dbm.edits.find_one({"_id": edit_id})
    if not edit:
        print(f"No edit found with ID: {edit_id}")
        return

    frames_dir = edit.get("frames_dir")
    frames_dir = osp.join(config["storage_dir"]["path"], frames_dir) if frames_dir else None
    if frames_dir and Path(frames_dir).exists():
        image_files = sorted(Path(frames_dir).glob("*.jpg"))
        image_map = {}
        for image_file in image_files:
            # Extract timestamp from filename, e.g. "frame_123.45.jpg" -> 123.45
            stem = image_file.stem  # "frame_123.45"
            if "_" in stem:
                timestamp_str = stem.split("_")[-1]
                try:
                    timestamp = float(timestamp_str)
                    image_map[timestamp] = str(image_file)
                except ValueError:
                    print(f"Could not parse timestamp from {image_file}")
    events = edit.get("events", [])

    subs_map = {}
    for event in events:
        command_name = event.get("commandName", "")
        if command_name in ["OK", "AutoSave"]:
            continue
        if command_name:
            subs_map[event["timestamp"]] = command_name  

    norm_images, (norm_subs,), t0, timeline_end = normalize_epoch_tracks_with_image_durations(
        image_map,
        [subs_map],
        speedup=1.0,
        min_image_duration=0.25,
        max_image_duration=0.25,
    )

    out_fn = osp.join(config["storage_dir"]["path"], "export", "frame_videos", f"edit_{edit['_id']}.mp4")
    Path(out_fn).parent.mkdir(parents=True, exist_ok=True)

    overlay_text = []
    request = dbm.requests.find_one({"_id": edit["request"]})
    if request:
        overlay_text.append(f"Request ID: {request['_id']}")
        overlay_text.append(f"Modality: {request.get('modality', 'N/A')}")
        overlay_text.append(f"Difficulty: {request.get('difficulty', 'N/A')}")
    overlay_text.append(f"Edit ID: {edit['_id']}")
    edit_duration = edit.get("end_time", 0) - edit.get("start_time", 0)
    # get edit duration in mins/seconds format. Use a time library
    mins = int(edit_duration // 60)
    secs = int(edit_duration % 60)
    edit_duration_str = f"{mins}m {secs}s"
    overlay_text.append(f"Edit Duration: {edit_duration_str}")

    overlay_text = "\n".join(overlay_text)


    dict_seconds_to_video(
        norm_images,
        out_fn,
        fps=30,
        tail_duration=2.0,
        speedup=1.0,
        subs_ts=norm_subs,
        width=1280,
        height=720,
        # min_subtitle_duration=1.0,
        # max_subtitle_duration=2.0,
        subtitle_duration=0.25,
        burn_in_text=overlay_text,
        burn_in_text_size=12,
    )      


# request ids from example video
request_ids = [
    # draw-static
    "F332D3FXML85WLR2_1770192609.750378",
    # draw-temp
    "B7A2N74ZJBF9MZHU_1770251861.898911",
    # interactive
    "3YH2WFSRM22W7DKT_1769160931.493333",
    # text
    "SUJ2G2UMJQR7PMBX_1759209987.785593",
]
edits = dbm.edits.find({"request": {"$in": request_ids}})



for edit in edits:
    edit_user = dbm.users.find_one({"_id": edit["user"]})
    is_human = edit_user.get("is_human", False) if edit_user else False
    if not is_human:
        # print(f"Skipping edit {edit['_id']} by non-human user {edit['user']}")
        continue

    create_video_from_edit_id(dbm, edit["_id"])
  

